Now, We have cleaned our dataset and create all usefull features that can be useful.

Let us move onto Exploratory Data Analysis (EDA)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

In [ ]:
## Importing dataset
Df = pd.read_parquet("../Dataset/Clean/dataset_with_features.parquet")
data = pd.read_parquet("../Dataset/Clean/BBC_with_features_combined.parquet")
print("Rows of kaggle:", len(Df))
print("Columns of kaggle:", len(Df.columns))
print("Rows of BBC:", len(data))
print("Columns of BBC:", len(data.columns))
Df.sample(3)

In [ ]:
## Related features need to be dropped to avoid multicollinearity & bias
df = Df.drop(columns=["ID","char_count_no_spaces","content_word_count",])

In [ ]:
# Optional interactive EDA (often unavailable on Kaggle / lightweight envs)
try:
    import dtale

    show = dtale.show(df)
    print(show._url)
    display(show, open_browser=False)
except Exception as e:
    print("dtale not available (skipping).", type(e).__name__, "-", str(e)[:200])

#### 1. Statistical summary

In [ ]:
# df_desc = df.describe()

# tbheader = dict(
#     values = list(df_desc.columns),
#     fill_color = 'lightgreen',
#     align = 'center',
#     line_color = 'black',
#     font = dict(size = 15, color = 'black', family = "Arial", weight = "bold")
# )
# tbcells = dict(
#     values = [df_desc[col] for col in df_desc.columns],
#     align = 'center', 
#     font = dict(size = 12, color = 'black', family = "Arial")
# )


# fig = go.Figure(data = [go.Table(
#     header = tbheader,
#     cells = tbcells,
#     )]
# )
# # Styling border
# fig.update_layout(
#     title = "Kaggle data Statistical Summary",
#     margin = dict(l = 10, r = 10, t = 40, b = 10),
#     template = "plotly_white",
#     width = 5000, height = 300
# )

# fig.show()

In [ ]:
# from IPython.display import HTML

# df_desc = df.describe()
# HTML("""
# <div style="max-height:400px; overflow:auto;">
# """ + df_desc.to_html() + """
# </div>
# """)

df_desc = df.describe().T
df_desc

For Kaggle dataset:
1. Average article has around 4124 words(large), 37 sentences with 670 words[284 unique words].
2. smallest article has 25 character - 1 sentence -> Article is very short.
3. largest article has 326910 characters - 2101 sentences -> Article is very long.
> Article has Outliers in terms of length.
4. Around 26-38% of word only appear once
5. Approximately 40% of article is just stopwords -> for grammatical structure. <br>
   So, <b>we can reduce article to 30%-60% for summary.</b>
6. There are 2 times more noun then adjective+adverb -> Article are not descriptive for all Objects.
7. Majority of article don't include any major event -> Sparsity issue.
8. Average Flesch score is 57.21 which is not high -> Around High school level
> Article are not beginner friendly to understand.
9. Flesch score below 0 or above 100 are not normal, need to check those article later.
   Similarily, kincaid grade should be b/w 0-12.
10. There are few article which are too diffcult or too long or too complex -> aka those need to be remove or treated.

#### 2. Column-Wise data analysis.

In [ ]:
## List of features
length_column = ['char_count','sentence_count','word_count','unique_word_count','stopword_count','stopword_ratio']
diversity_column = ['lexical_diversity','hapax_ratio']
pos_column = ['noun_count', 'verb_count', 'adj_count', 'adv_count', 'pronoun_count']
ner_column = ['person_count', 'org_count', 'gpe_count', 'event_count', 'unique_entity_count']
reading_ease = ['flesch_reading_ease', 'flesch_kincaid_grade', 'gunning_fog']

data = df.copy()

In [ ]:
## 1) Source distribution
src_dist = df['Dataset'].value_counts()

plot_df = (
    src_dist
    .reset_index()
    .rename(columns={"index": "Dataset"})
)
plot_df["percent"] = (plot_df["count"] / plot_df['count'].sum()) *100

fig = px.bar(
    plot_df,
    x="Dataset",
    y="count",
    text=plot_df["percent"].round(2).astype(str) + "%",
    title="Source distribution",
    template="plotly_white",
)
fig.update_traces(textposition="inside", textfont_size=12)
fig.update_layout(xaxis_tickangle=-45)
fig

In [ ]:
import scipy

# Mark all numerical outliers
all_outlier_indices = set()

## 2) Length wise analysis
char_count = df['char_count']
log_char_count = np.log(char_count)

print("Mean:", log_char_count.mean())
print("Std Dev:", log_char_count.std())

In [ ]:
fig,axes = plt.subplots(1,2, figsize=(12,5))

# Histogram+KDE
sns.histplot(log_char_count, bins=20, kde=True,
    ax = axes[0], color='coral', edgecolor='black')
axes[0].set_title("Log Character Count Distribution")
axes[0].grid(True)
# QQ plot
scipy.stats.probplot(log_char_count, dist="norm", plot=axes[1])
axes[1].set_title("QQ Plot for log char count")
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
z_score = scipy.stats.zscore(log_char_count)
outliers = df[np.abs(z_score) > 3]

print("Number of outliers:", len(outliers))
all_outlier_indices.update(outliers.index)

In [ ]:
## Group all numerical columns
count_cols = ["word_count", "sentence_count", "unique_word_count", "stopword_count"]

n_cols = len(count_cols)
fig, axes = plt.subplots(
    n_cols, 2,
    figsize=(14, 4 * n_cols),
    squeeze=False
)

for i, col in enumerate(count_cols):
    # Log transform (avoid log(0))
    log_values = np.log(df[col].replace(0, np.nan)).dropna()

    # Stats
    mean = log_values.mean()
    std = log_values.std()
    print(f"{col} → Mean: {mean:.3f}, Std: {std:.3f}")

    # Histogram + KDE
    sns.histplot(log_values, bins=20, kde=True, ax=axes[i, 0], color='firebrick')
    axes[i, 0].set_title(f"Log {col} Distribution")
    axes[i, 0].grid(True)

    # QQ plot
    scipy.stats.probplot(log_values, dist="norm", plot=axes[i, 1])
    axes[i, 1].set_title(f"QQ Plot for Log {col}")
    axes[i, 1].grid(True)

    # Z-score outliers
    z_scores = np.abs(scipy.stats.zscore(log_values))
    outlier_idx = log_values.index[z_scores > 3]

    all_outlier_indices.update(outlier_idx)

plt.tight_layout()
plt.show()


- Majority of News Article comes from CNN. So, we need to check for any bias/distortion in CNN news.
- Char count is Log-Normally distributed (as often observed)
- Similarily, [sentence count, word count, unique word count, stopword_count] is also Log-Normally distributed. <br>
  -> All numerical features are Log-Normally distributed.

In [ ]:
## 3) Quality ratio analysis
lexdiv = df["lexical_diversity"]
print("Skew of Lexical Diversity:", scipy.stats.skew(lexdiv))

fig,axes = plt.subplots(1,2, figsize=(12,5))

# Histogram+KDE
sns.histplot(lexdiv, bins=50, kde=True,
    ax = axes[0], color='aqua', edgecolor='black')
axes[0].set_title("Lexical Diversity Dist")
axes[0].grid(True)
# QQ plot
scipy.stats.probplot(lexdiv, dist="norm", plot=axes[1])
axes[1].set_title("QQ Plot for Lexical Diversity")
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
hapaxdiv = df["hapax_ratio"]
print("Skew of hapax ratio Diversity:", scipy.stats.skew(hapaxdiv))

fig,axes = plt.subplots(1,2, figsize=(12,5))

# Histogram+KDE
sns.histplot(hapaxdiv, bins=50, kde=True,
    ax = axes[0], color='darkgreen', edgecolor='black')
axes[0].set_title("Hapax ratio Dist")
axes[0].grid(True)
# QQ plot
scipy.stats.probplot(hapaxdiv, dist="norm", plot=axes[1])
axes[1].set_title("QQ Plot for Hapax ratio")
axes[1].grid(True)

plt.tight_layout()
plt.show()

- Lexical diversity is almost normally distributed -> skew = 0.88
- hapax ratio has less normality. It is right-skewed -> skew = 1.3 

In [ ]:
## 4) Grammar profile analysis
descriptiveness = (df['adj_count']+df['adv_count']) / df['word_count']
action_ratio = (df['verb_count'] / df['word_count'])
Objective_ratio = (df['noun_count']+df['pronoun_count']) / df['word_count']

In [ ]:
## A] POS Profile
# 1. Normalize POS columns by word_count to get percentages
df_pos_pct = df[pos_column].div(df['word_count'], axis=0) * 100
# 2. Melt for plotting -> Convert wide to long format
melted_pos = df_pos_pct.melt(var_name='POS_Type', value_name='Percentage') 

## result: each type for each article form each row with percentage value
print(melted_pos.groupby('POS_Type')['Percentage'].agg(['mean', 'median']))

fig1 = px.box(
    melted_pos, 
    x='POS_Type', 
    y='Percentage', 
    color='POS_Type',
    title='Grammar Profile: Distribution of Word Types (%) per Document',
    template='plotly_white'
)
fig1.show()

- Majority of article have around 11.9%-30.9% noun words, 8.1%-18.8% verb words, 1.6%-11.8% adjectives, 0%-7.3% adverbs and 0%-16.1% pronouns words.
- POS Occurance: Noun > Verb > Pronoun > Adjective > Adverb
- Percentage of Noun is very highly variable.
- Average Percentage by POS tag: <br>
  21.3% Noun, 13.4% Verb, 6.7% Adjective, 3.5% Adverb, 6.9% Pronoun

In [ ]:
## B] Verb to Noun Ratio Analysis
verb_noun_ratio = pd.Series(np.where(df['noun_count'] > 0, df['verb_count'] / df['noun_count'], 0)) # ratio (handle potential 0 nouns)

print("Verb to Noun Ratio - Mean:", verb_noun_ratio.mean())
print("Verb to Noun Ratio - Standard Deviation:", verb_noun_ratio.std())
print("Verb to Noun Ratio - Skew:", verb_noun_ratio.skew())
print("Verb to Noun Ratio - Kurtosis:", verb_noun_ratio.kurtosis())

fig2 = px.histogram(
    x=verb_noun_ratio, 
    nbins=50,
    marginal='box', # Adds a box plot on top
    title='Distribution of Verb-to-Noun Ratio (Action vs. Description)',
    labels={'verb_noun_ratio': 'Ratio (Higher = More Verbs)'},
    template='plotly_white',
    color_discrete_sequence=['#636EFA']
)
fig2.show()

- Verb to Noun ratio is right-skewed normally distributed[skew = 0.91, kurtosis = 2.43].
- Most articles have around 1.5 noun to verb ratio. On average, articles are more descriptive than action oriented.

In [ ]:
## C] Fluff index - descriptiveness
fig3 = px.histogram(
    x = descriptiveness, nbins = 50,
    marginal='box',
    title = "Dist of Descriptiveness Index (Adj+Adv per word)",
    labels = {'descriptiveness': 'Descriptiveness Index'},
    template='plotly_white',
    color_discrete_sequence=['#EF553B']
)
fig3.show()

- Around 10% of article is about descriptiveness.
- All pos count are left-skewed log-normally distributed. Log[pos] is almost +vely correlated with each other.
- Multi-News article are more grammatically charged (more POS) then CNN, XSum.

In [ ]:
## D] Pairwise POS relationships
# Prepare the data: Log-transform the counts to reduce skewness
# We use log1p (log(1+x)) to avoid errors with zero counts
df_log = df.copy()
for col in pos_column:
    df_log[f'log_{col}'] = np.log1p(df[col])

# 3. Create the Scatter Matrix
fig = px.scatter_matrix(
    df_log,
    dimensions=[f'log_{col}' for col in pos_column],
    color='Dataset',               # Differentiate datasets by color
    symbol='Dataset',              # Differentiate by shape (useful for overlap)
    opacity=0.5,                   # Add transparency to see density
    title='Linguistic Fingerprints: Pairplot of Log-Transformed POS Counts',
    labels={f'log_{col}': col.replace('_count', '') for col in pos_column},
    template='plotly_white'
)

# 4. Beautify: Update marker size and layout
fig.update_traces(diagonal_visible=False, marker=dict(size=4))
fig.update_layout(
    width=1000, 
    height=1000,
    dragmode='select', # Allows you to highlight specific clusters
    hovermode='closest'
)

fig.show()

In [ ]:
## 5) Entity Composition
## A] Overall entity composition
entity_sums = df[ner_column].agg(['mean','sum']).T.reset_index()
entity_sums.columns = ['Entity_type', 'Mean_count', 'Total_count']

fig4 = px.pie(
    entity_sums,
    values = 'Mean_count',
    names='Entity_type', hole=0.4,
    title="Overall Entity Composition [Who/What/Where]",
    template='ggplot2',
    color_discrete_sequence = px.colors.qualitative.Pastel,
    height=400, width=700
)
fig4.show()

In [ ]:
## B] Entity Sparsity
sparsity = (df[ner_column] != 0).sum() / len(df)
sparsity = sparsity.reset_index()
sparsity.columns = ['Entity_type', 'Sparsity_percent']

fig5 = px.bar(
    sparsity,
    x='Entity_type', y='Sparsity_percent',
    title='Entity Sparsity Across Documents (%)',
    template='seaborn',
    color_discrete_sequence=['#AB63FA']
)
fig5.show()

Out of all Recognized named entities,
- 42.9% are person, 31.1% are org, 24.9% are location and 1.06% are event.
- There are very few famous recognizable event -> Event are either not mentioned or can't be recognized.

In [ ]:
## Outliers:
print("Total numerical outliers identified:", len(all_outlier_indices))

# dropping article with large amount of outliers
df = df.drop(index=all_outlier_indices)

#### 3. Relation b/w Columns:

In [ ]:
# A) Correlation Analysis
corr = df.corr(numeric_only=True).round(3)

corr_plot = px.imshow(corr, color_continuous_scale='RdBu_r', text_auto=True, 
    aspect='auto', height=1000, width=2000)
corr_plot.show()

- Similar type of features are highly correlated [note for prediction]
- As length of article increase, all length & count based features all increase [+ve corr]
- Qualitative features like gunning fog, reading-ease are independent to Quantative features like sentence_count
- Lexical diversity is inversely prop. to pos count -> Diversity in vocabulary lead to less gramatical sense.
- More lexical diversity lead to more entity density.

In [ ]:
df_summary = pd.read_parquet("../Dataset/Clean/dataset_summary_features.parquet")
df_summary = df_summary.drop(columns=['ID','char_count_no_spaces','content_word_count'])

print(df.columns)
print(df_summary.columns)

In [ ]:
Length = pd.merge(left = df, right=df_summary, on =['Content','Summary','Dataset'], how ='inner', suffixes=('', '_summary'))


Length['Log_content_char_count'] = np.log1p(Length['char_count'])
Length['Log_summary_char_count'] = np.log1p(Length['char_count_summary'])

Length['word_ratio'] = Length['word_count_summary'] / Length['word_count']
Length['char_ratio'] = Length['char_count_summary'] / Length['char_count']
Length['sent_ratio'] = Length['sentence_count_summary'] / Length['sentence_count']

Length.head(2)

In [ ]:
# B] Length wise features -> Create the Scatter Matrix
scaled_length = Length[length_column]

for col in length_column:
    std = scaled_length[col].std()
    mean = scaled_length[col].mean()
    min_ = scaled_length[col].min()
    max_ = scaled_length[col].max()
    print(f"{col} → Mean: {mean:.3f}, Std: {std:.3f}, Min: {min_}, Max: {max_}")
    
    scaled_length[col] = (scaled_length[col] - min_) / (max_ - min_)
    

# Create the Scatter Matrix
fig = px.scatter_matrix(
    scaled_length,
    opacity=0.5,                   # Add transparency to see density
    title='Linguistic Fingerprints: Pairplot of Standardized Length Features',
    labels={f'{col}': col.replace('_count', '') for col in length_column},
    template='ggplot2',
    
)
fig.update_traces( marker=dict(size=4))
fig.update_layout(
    height=1000, width=1000, 
    dragmode='select', # Allows you to highlight specific clusters
    hovermode='closest'
)

fig.show()

In [ ]:
# C] Content length vs Summary length
fig3 = px.scatter(
    Length, 
    x="Log_content_char_count", 
    y="Log_summary_char_count",
    marginal_x="histogram", 
    marginal_y="histogram",
    title="Log-Scale Correlation: Content vs. Summary (Characters)",
    labels={'Log_content_char_count': 'Log(Content)', 'Log_summary_char_count': 'Log(Summary)'},
    template="plotly_white",
    color_discrete_sequence=['#EF553B']
)
fig3.show()

- Length features to stopword ratio are left-skewed beta distributed.
- Mostly linear relation between char count and sentence count
- There are some article where summary is longer then content.
- Content and Summary length has a small linear link -> almost independent

In [ ]:
# D] Melting ratios for a combined distribution plot
ratio_cols = ['word_ratio', 'char_ratio', 'sent_ratio']
melted_ratios = Length.melt(value_vars=ratio_cols, var_name='Ratio_Type', value_name='Value')

fig1 = px.histogram(
    melted_ratios, 
    x="Value", 
    color="Ratio_Type", 
    marginal="box", 
    barmode="overlay",
    title="Distribution of Compression Ratios (Summary / Content)",
    labels={'Value': 'Ratio (0.1 = 10% of original size)'},
    template="plotly_white",
    opacity=0.7
)
fig1.show()

- Article where summary is longer then article are kinda invalid. Need to remove them.
- On average, the summary is 10% smaller than the content

In [ ]:
longer_summ = Length[(Length['sent_ratio'] > 1) | (Length['char_ratio'] > 1) | (Length['word_ratio'] > 1)]
df = df.drop(index=set(longer_summ.index))
Length = Length.drop(index=set(longer_summ.index))
print(Length.shape)

In [ ]:
# E] Content length vs Summary length
print("Average Reduction Ratios:")
print(f"Word Ratio: {1/Length['word_ratio'].median():.2f}")
print(f"Char Ratio: {1/Length['char_ratio'].median():.2f}")
print(f"Sent Ratio: {1/Length['sent_ratio'].median():.2f}")

fig = px.histogram(
    Length,
    x="char_ratio",
    nbins=50,
    marginal='box',
    title="Dist of Summary Reduction Ratio (Summary sentences / Content sentences)",
    labels={'char_ratio': 'Reduction Ratio'},
    template='plotly_white',
    color_discrete_sequence=['#00CC96']
)
fig.show()

In [ ]:
# F] Entity retention ratio
print("Entity average for content:", Length['unique_entity_count'].median())
print("Entity average for summary:", Length['unique_entity_count_summary'].median())
print("Entity Retention - Median:", Length['unique_entity_count_summary'].median() / Length['unique_entity_count'].median())
print("Entity Retention - Mean:", Length['unique_entity_count_summary'].mean() / Length['unique_entity_count'].mean())

fig6 = px.scatter(
    Length, 
    x="unique_entity_count_summary", 
    y="unique_entity_count",
    marginal_x="histogram", 
    marginal_y="histogram",
    title="Entity Retention: Summary vs. Content",
    labels={'unique_entity_count_summary': 'Unique Entities in Summary', 'unique_entity_count': 'Unique Entities in Content'},
    template="plotly_white",
    color_discrete_sequence=['#AB63FA']
)
fig6.show()

- There are article with more entity in summary which is not normal.
- There doesn't seem to be any relationship b/w entity count of content and summary.
- On average, there are 7.4 times more entity in content then summary.

In [ ]:
Length.query('unique_entity_count_summary/unique_entity_count > 1')[['Content','Summary','unique_entity_count','unique_entity_count_summary']]

> Summary has entities in recognisable format, not more entity. <br>
For Better recognition, we need to standardize the entity format or use something better for entity extraction.

In [ ]:
# G) Entity Density vs Readibility:
fig7 = px.scatter(
    Length, 
    x=Length['unique_entity_count'] / Length['word_count'], 
    y='flesch_kincaid_grade', 
    trendline='ols', # Adds a linear regression line
    opacity=0.4,
    title='Impact of Entity Density on Reading Difficulty',
    template='plotly_white'
)
fig7.update_layout(xaxis_title='Entities per Word', yaxis_title='Reading Grade Level')
fig7.show()

In [ ]:
# H] Ideal 'Compression' Ratios
fig4 = px.scatter(Length, x='word_ratio', y='flesch_reading_ease', trendline="lowess",
    title="Q4: Readability vs. Compression Ratio",
    labels={'word_ratio': 'Compression Ratio (Sum/Cont)', 'flesch_reading_ease': 'Reading Ease'},
    template="plotly_white"
)
fig4.show()

In [ ]:
# Removing outlier for reading ease
Length = Length[Length['flesch_reading_ease'] > 0]

- Reading Level is independent of entity density/presence.
- Readability is independent of compression_ratio
- Readability also doesn't seem to be affected by Grammar profile.

In [ ]:
# I] Predicting readbility from grammar profile
corr_matrix = Length[['flesch_reading_ease']+ner_column].corr()
fig = px.imshow(corr_matrix.round(4), text_auto=True, title="Correlation: Grammar Profile vs Reading Ease")
fig.update_layout(width=800, height=600)
fig.show()

In [ ]:
# J] Lexical Diversity for Content vs Summary
fig8 = go.Figure()
fig8.add_trace(go.Violin(y=Length['lexical_diversity'], name='Content Diversity', box_visible=True, meanline_visible=True))
fig8.add_trace(go.Violin(y=Length['lexical_diversity_summary'], name='Summary Diversity', box_visible=True, meanline_visible=True))
fig8.update_layout(title='Lexical Diversity: Content vs Summary')
fig8.show()

- Summary have higher lexical diversity -> Content is skewed toward higher diversity while summary is skewed towards lower diversity

In [ ]:
# K] Narritive vs Hard News comparison
fig9 = px.histogram(x=Length['pronoun_count']/(Length['noun_count']), nbins=50,
    title="Narrative vs Hard News: Pronoun to Noun Ratio Distribution",
    labels={'x': 'Pronoun to Noun Ratio'},
    template='plotly_white',
    color_discrete_sequence=['#FFA15A']
)
fig9.show()

- Dataset has way more noun to pronoun -> Article are most written in formal voice.

In [ ]:
# L] Dataset-wise Source-wise comparison
if "Dataset" not in df.columns:
    print("No 'Dataset' column; skipping dataset-wise comparison.")
else:
    compare_cols = [c for c in ["word_count", "sentence_count", "lexical_diversity", "flesch_reading_ease", "gunning_fog"] if c in df.columns]
    if not compare_cols:
        print("No common comparison columns found.")
    else:
        # Violin plots (good for distribution comparisons)
        for c in compare_cols:
            fig = px.violin(
                df,
                x="Dataset",
                y=c,
                box=True,
                points=False,
                title=f"{c} by dataset",
                template="plotly_white",
            )
            fig.update_layout(xaxis_tickangle=-20)
            fig.show()

        # Group stats table
        grp = df.groupby("Dataset")[compare_cols].agg(["mean", "median", "std"]).round(3)
        display(grp)

- Some article from CNN & Xsum are way too advance/complex compared to Multi-News. On average, Multi-News are more beginner friendly.
- CNN+XSum articles have higher lexical diversity.
- Length-wise: Multi-News > CNN > XSum.

In [ ]:
# Saving combined Length dataframe for future use
Length.to_parquet("../Dataset/Clean/Combined_dataset.parquet", index=False)